# 04 · Baselines & LightGBM (skeleton)

The ablation ladder:

| ID | Model | Features |
|---|---|---|
| A | Persistence | y(t) |
| B | CAMS issue-time forecast | raw forecast for t+h |
| C | LightGBM | weather + CAMS (+ PM2.5 lags) |
| D | LightGBM | C + hotspot counts |
| E | LightGBM | D + **upwind exposure** |

`REAL_DATA_READY` stays `False` until notebook 02/03 produce a real, aligned table in
`ml/data/processed/`. **Do not** run model training on the synthetic fixture and report the numbers.

In [ ]:
from pavana_ml.paths import PROCESSED_DIR

TABLE = PROCESSED_DIR / "model_table.parquet"
REAL_DATA_READY = TABLE.exists()
TARGET = "target_pm25_ug_m3_t+6h"
print("real data table:", TABLE, "exists:", REAL_DATA_READY)

In [ ]:
if REAL_DATA_READY:
    import pandas as pd

    from pavana_ml.evaluation.metrics import event_precision_recall, mae, rmse
    from pavana_ml.evaluation.splits import temporal_split
    from pavana_ml.models import baselines, lgbm

    df = pd.read_parquet(TABLE).dropna(subset=[TARGET]).sort_values("valid_at")
    train, test = temporal_split(
        df, "valid_at", test_start="2023-10-01", gap_hours=6
    )  # TODO choose split

    FEATURES = {
        "C": [
            "cams_pm25_t+6h",
            "wind_speed_ms",
            "rh_pct",
            "precip_mm",
            "pm25_ug_m3_lag1h",
            "pm25_ug_m3_lag3h",
        ],
    }
    FEATURES["D"] = FEATURES["C"] + ["hotspots_n"]
    FEATURES["E"] = FEATURES["D"] + ["upwind_score"]

    rows = [
        {
            "model": "A persistence",
            "MAE": mae(test[TARGET], baselines.persistence(test, "pm25_ug_m3")),
        },
        {
            "model": "B CAMS",
            "MAE": mae(test[TARGET], baselines.cams_issue_time(test, "cams_pm25_t+6h")),
        },
    ]
    for key, cols in FEATURES.items():
        model = lgbm.train(train[cols], train[TARGET])
        pred = model.predict(test[cols])
        rows.append(
            {
                "model": f"{key} LightGBM",
                "MAE": mae(test[TARGET], pred),
                "RMSE": rmse(test[TARGET], pred),
                **event_precision_recall(test[TARGET], pred, 55.5),
            }
        )
    display(pd.DataFrame(rows))
else:
    print("Skipping: no real model table yet. See notebooks 02 and 03.")